In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
# Imports 
from transformers import  Gemma2ForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback, BitsAndBytesConfig
from datasets import Dataset
import pandas as pd
import numpy as np
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
import torch
import re
from pathlib import Path
from kaggle_secrets import UserSecretsClient

In [3]:
import shutil
shutil.copytree(
    "/kaggle/input/datasets/rpatel29/gemma-args-2/checkpoint-2400",
    "/kaggle/working/checkpoint-2400"
)


'/kaggle/working/checkpoint-2400'

In [4]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret('HF_TOKEN')

# Setup configuration to lower load on VRAM
config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load the model's tokenizer and the pre-trained model
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b-it", token=secret_value_0)
tokenizer.pad_token = tokenizer.eos_token

base_model = Gemma2ForCausalLM.from_pretrained("google/gemma-2-9b-it", 
                                               quantization_config=config, 
                                               device_map="auto", 
                                               token=secret_value_0
                                              )

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [5]:
# Load the training and validation data
train_data = pd.read_csv('/kaggle/input/datasets/rpatel29/chatbot-data/completed-data-2/train/train.csv')
val_data = pd.read_csv('/kaggle/input/datasets/rpatel29/chatbot-data/completed-data-2/val/val.csv')

In [ ]:
"""
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

# Apply LoRA to ensure memory efficient training
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] 
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()
"""

In [6]:

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/checkpoint-2400",
    is_trainable=True,
    local_files_only=True,
)

model.train()



PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
           

In [7]:
max_length = 896

# Function to preprocess the data for training
def preprocess(example):

    user_message = (
        f"Task: {example['task_type']}\n"
        f"Instruction: {example['instruction']}\n"
        f"Patient: {example['input']}"
    )

    messages = [
        {
            "role": "user",
            "content": user_message
        },
        {
            "role": "assistant",
            "content": example['output']
        }
    ]

    # Apply the chat template and then tokenize it
    full_message = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(full_message, truncation=True, max_length=max_length, padding=False)

    user_only = [
        {
            "role": "user",
            "content": user_message
        }

    ]

    user_text = tokenizer.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    user_tokens = tokenizer(user_text, add_special_tokens=False)['input_ids']

    
    num_tokens = len(user_tokens)

    label = ([-100] * num_tokens + tokenized['input_ids'][num_tokens:])
    label = label[:len(tokenized["input_ids"])]

    tokenized["labels"] = label

    return tokenized
    

    


In [8]:
from dataclasses import dataclass

# Creates a custom data collator 
@dataclass
class CustomDataCollator:
    tokenizer: any

    def __call__(self, features):
        input_ids = [feature['input_ids'] for feature in features]
        attention_mask = [feature['attention_mask'] for feature in features]
        labels = [feature['labels'] for feature in features]

        batch = {
            "input_ids": torch.nn.utils.rnn.pad_sequence(
                [torch.tensor(x, dtype=torch.long) for x in input_ids],
                batch_first=True,
                padding_value=self.tokenizer.pad_token_id
            ),

            "attention_masks": torch.nn.utils.rnn.pad_sequence(
                [torch.tensor(x, dtype=torch.long) for x in attention_mask],
                batch_first=True,
                padding_value=0
            ),

            "labels": torch.nn.utils.rnn.pad_sequence(
                [torch.tensor(x, dtype=torch.long) for x in labels],
                batch_first=True,
                padding_value=-100
            )
            
        }

        return batch 
        
        
    

In [9]:
data_collator = CustomDataCollator(
    tokenizer=tokenizer
)

In [10]:
# Prepare data for training 
#train_data = train_data[~train_data["output"].apply(is_invalid)]
#val_data = val_data[~val_data["output"].apply(is_invalid)]

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)

train_dataset = train_dataset.remove_columns(
    ["task_type", "instruction", "input", "output"]
)

val_dataset = val_dataset.remove_columns(
    ["task_type", "instruction", "input", "output"]
)

Map:   0%|          | 0/40084 [00:00<?, ? examples/s]

Map:   0%|          | 0/5038 [00:00<?, ? examples/s]

In [ ]:
example = train_dataset[0]

print("Input length:", len(example["input_ids"]))
print("Label length:", len(example["labels"]))

print("Ignored:", example["labels"].count(-100))
print("Trainable:", sum(x != -100 for x in example["labels"]))

print("\nINPUT:")
print(tokenizer.decode(example["input_ids"], skip_special_tokens=False))

print("\nLABELS:")
print(tokenizer.decode(
    [x for x in example["labels"] if x != -100],
    skip_special_tokens=False
))

In [ ]:
example = train_dataset[0]

batch = data_collator([example])

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)

labels = batch["labels"][0]

print("Total tokens:", len(labels))
print("Ignored (-100):", (labels == -100).sum().item())
print("Trainable:", (labels != -100).sum().item())

print("\nLABEL IDS:")
print(labels.tolist())

print("\nDECODED LABELS:")
print(tokenizer.decode(
    [x for x in labels.tolist() if x != -100],
    skip_special_tokens=False
))

In [ ]:
example = train_dataset[0]

i = tokenizer.decode(example["input_ids"], skip_special_tokens=False)
print(i)

In [11]:
class PrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step={state.global_step}, logs={logs}")

In [12]:
# Setup the arguments for training
training_args = TrainingArguments(
    output_dir="/kaggle/working/gemma_2b_chatbot_v1/",
    
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    
    learning_rate=5e-6,
    num_train_epochs=1,
    
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    greater_is_better=False,

    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,

    save_steps=400,
    eval_steps=200,
    save_total_limit=3,
    
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=20,
    max_grad_norm=1.0,
    remove_unused_columns=False,
    report_to="none",
    
    fp16=True,
    bf16=False,
)

In [ ]:
# Trains the model and validates using validation set
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[PrinterCallback],
)

trainer.train(resume_from_checkpoint='/kaggle/working/checkpoint-2400')

Step,Training Loss,Validation Loss
2600,1.128530,0.980544


step=2420, logs={'loss': 1.1310750007629395, 'grad_norm': 3.883139133453369, 'learning_rate': 3.831267009374056e-06, 'epoch': 0.2414928649835346}
step=2440, logs={'loss': 1.2217172622680663, 'grad_norm': 5.273203372955322, 'learning_rate': 3.821187380304405e-06, 'epoch': 0.2434886737850514}
step=2460, logs={'loss': 1.1717561721801757, 'grad_norm': 4.758499622344971, 'learning_rate': 3.811107751234755e-06, 'epoch': 0.24548448258656821}
step=2480, logs={'loss': 1.0920434951782227, 'grad_norm': 5.705284595489502, 'learning_rate': 3.8010281221651048e-06, 'epoch': 0.24748029138808503}
step=2500, logs={'loss': 1.1252070426940919, 'grad_norm': 4.043618202209473, 'learning_rate': 3.7909484930954543e-06, 'epoch': 0.24947610018960184}
step=2520, logs={'loss': 1.1806300163269043, 'grad_norm': 4.017075061798096, 'learning_rate': 3.780868864025804e-06, 'epoch': 0.25147190899111865}
step=2540, logs={'loss': 1.2407170295715333, 'grad_norm': 5.104633331298828, 'learning_rate': 3.7707892349561537e-06, 

In [ ]:
import os
import glob


for zip_file in glob.glob("/kaggle/working/*.zip"):
    os.remove(zip_file)
    print(f"Deleted: {zip_file}")

In [14]:
import shutil

# Folder containing checkpoints
source_dir = "/kaggle/working/gemma_2b_chatbot_v1"

# Zip entire training directory
shutil.make_archive(
    "/kaggle/working/gemma_2b_chatbot_v3",
    "zip",
    source_dir
)

print("Backup zip created!")

Backup zip created!


In [ ]:
!nvidia-smi